# Banking Customer & Loan Analytics

## Notebook 2: Pandas Data Cleaning

### Project Objective

This notebook focuses on cleaning and preparing the raw banking
datasets for further exploratory data analysis.

The datasets include:

- Customers
- Accounts
- Transactions
- Loans
- Branches

### Cleaning Objectives

1. Load raw datasets
2. Create separate working copies
3. Inspect missing values
4. Handle duplicate customer records
5. Correct data types
6. Standardize dates
7. Standardize city values
8. Standardize account types
9. Identify invalid transaction amounts
10. Identify unusual loan values
11. Check relationships between tables
12. Save cleaned datasets separately

### Important Principle

The original files inside `data/raw_dataset/` will never be modified.

All cleaning operations will be performed on working DataFrames,
and the final results will be saved inside `data/cleaned_dataset/`.

## 2. Import Required Libraries

Pandas will be the primary library used for data cleaning.

NumPy will be used where numerical operations are required.

In [ ]:
import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


## 3. Load Raw Banking Datasets

We will load the original datasets from the raw dataset folder.

No changes will be made to these files during the cleaning process.

In [ ]:
customers_raw = pd.read_csv(
    "../data/raw_dataset/customers.csv"
)

accounts_raw = pd.read_csv(
    "../data/raw_dataset/accounts.csv"
)

transactions_raw = pd.read_csv(
    "../data/raw_dataset/transactions.csv"
)

loans_raw = pd.read_csv(
    "../data/raw_dataset/loans.csv"
)

branches_raw = pd.read_csv(
    "../data/raw_dataset/branches.csv"
)

print("All raw datasets loaded successfully.")

All raw datasets loaded successfully.


## 4. Create Working Copies

To protect the original raw datasets, we will create independent
working copies.

All cleaning operations will be performed on these copies.

In [ ]:
customers = customers_raw.copy()
accounts = accounts_raw.copy()
transactions = transactions_raw.copy()
loans = loans_raw.copy()
branches = branches_raw.copy()

print("Working copies created successfully.")

Working copies created successfully.


## 5. Check Initial Dataset Shape

We will record the initial number of rows and columns before cleaning.

This allows us to compare the dataset size before and after cleaning.

In [ ]:
print("Customers:", customers.shape)
print("Accounts:", accounts.shape)
print("Transactions:", transactions.shape)
print("Loans:", loans.shape)
print("Branches:", branches.shape)

Customers: (1002000, 7)
Accounts: (1300000, 6)
Transactions: (5000000, 7)
Loans: (500000, 8)
Branches: (50, 5)


## 6. Create Dataset Dictionary

A dictionary allows us to process all banking datasets using loops.

This will make the cleaning workflow reusable and organized.

In [ ]:
datasets = {
    "Customers": customers,
    "Accounts": accounts,
    "Transactions": transactions,
    "Loans": loans,
    "Branches": branches
}

print("Dataset dictionary created successfully.")

Dataset dictionary created successfully.


## 7. Missing Value Audit

Missing values can affect calculations and business analysis.

We will identify the number and percentage of missing values
in every column before deciding how they should be handled.

In [ ]:
def missing_value_report(df):
    report = pd.DataFrame({
        "Missing Count": df.isna().sum(),
        "Missing Percentage": (
            df.isna().mean() * 100
        ).round(2)
    })
    
    return report.sort_values(
        "Missing Count",
        ascending=False
    )

In [ ]:
for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    display(missing_value_report(df))


Customers


,Missing Count,Missing Percentage
Occupation,2004,0.2
City,1003,0.1
Income,1000,0.1
Gender,0,0.0
Age,0,0.0
CustomerID,0,0.0
JoinDate,0,0.0



Accounts


,Missing Count,Missing Percentage
AccountID,0,0.0
CustomerID,0,0.0
AccountType,0,0.0
Balance,0,0.0
OpenDate,0,0.0
Branch,0,0.0



Transactions


,Missing Count,Missing Percentage
TransactionID,0,0.0
AccountID,0,0.0
TransactionDate,0,0.0
TransactionType,0,0.0
Amount,0,0.0
Channel,0,0.0
Branch,0,0.0



Loans


,Missing Count,Missing Percentage
LoanID,0,0.0
CustomerID,0,0.0
LoanType,0,0.0
LoanAmount,0,0.0
InterestRate,0,0.0
LoanDate,0,0.0
LoanStatus,0,0.0
RepaymentAmount,0,0.0



Branches


,Missing Count,Missing Percentage
BranchID,0,0.0
BranchName,0,0.0
City,0,0.0
ManagerExperienceYears,0,0.0
OpenDate,0,0.0


## 8. Handle Missing Customer Values

The customer dataset contains important demographic and financial
information.

We will inspect missing values in:

- City
- Occupation
- Income

Before filling missing values, we will preserve the original
missing-value counts for comparison.

In [ ]:
customer_missing_before = {
    "City": customers["City"].isna().sum(),
    "Occupation": customers["Occupation"].isna().sum(),
    "Income": customers["Income"].isna().sum()
}

print("Missing values before treatment:")
customer_missing_before

Missing values before treatment:


{'City': np.int64(1003),
 'Occupation': np.int64(2004),
 'Income': np.int64(1000)}

## 9. Handle Missing City Values

City is a categorical variable.

For missing city values, we will use the label `"Unknown"` instead
of inventing a city.

This preserves the record while clearly indicating unavailable data.

In [ ]:
customers["City"] = customers["City"].fillna("Unknown")

print(
    "Remaining missing City values:",
    customers["City"].isna().sum()
)

Remaining missing City values: 0


## 10. Handle Missing Occupation Values

Occupation is a categorical variable.

Missing occupation values will be represented as `"Unknown"`.

This avoids assigning an unsupported occupation to a customer.

In [ ]:
customers["Occupation"] = customers["Occupation"].fillna(
    "Unknown"
)

print(
    "Remaining missing Occupation values:",
    customers["Occupation"].isna().sum()
)

Remaining missing Occupation values: 0


## 11. Handle Missing Income Values

Income is a numerical variable.

We will not replace missing income values with an arbitrary number.

Instead, missing income values will remain identifiable as missing
during this cleaning stage unless a justified business rule is available.

This prevents introducing artificial income values.

In [ ]:
print(
    "Remaining missing Income values:",
    customers["Income"].isna().sum()
)

Remaining missing Income values: 1000


## 12. Identify Duplicate Customer Records

Duplicate customer records can cause incorrect customer counts
and distort customer-level analysis.

We will identify duplicate CustomerID values before removing them.

In [ ]:
duplicate_customer_ids = customers[
    customers["CustomerID"].duplicated(keep=False)
]

print(
    "Rows involving duplicate CustomerIDs:",
    duplicate_customer_ids.shape[0]
)

print(
    "Duplicate CustomerIDs:",
    customers["CustomerID"].duplicated().sum()
)

Rows involving duplicate CustomerIDs: 4000
Duplicate CustomerIDs: 2000


## 13. Remove Duplicate Customer Records

CustomerID should represent a unique customer.

We will retain the first occurrence of each CustomerID and remove
subsequent duplicate records.

The raw customer dataset remains untouched.

In [ ]:
customers = customers.drop_duplicates(
    subset="CustomerID",
    keep="first"
).copy()

print(
    "Remaining duplicate CustomerIDs:",
    customers["CustomerID"].duplicated().sum()
)

print(
    "Customers after duplicate removal:",
    customers.shape[0]
)

Remaining duplicate CustomerIDs: 0
Customers after duplicate removal: 1000000


## 14. Correct Customer Data Types

We will convert numerical customer columns to appropriate
numeric data types.

Date standardization will be handled separately.

In [ ]:
customers["Age"] = pd.to_numeric(
    customers["Age"],
    errors="coerce"
)

customers["Income"] = pd.to_numeric(
    customers["Income"],
    errors="coerce"
)

print(customers[["Age", "Income"]].dtypes)

Age         int64
Income    float64
dtype: object


## 15. Standardize Customer Join Date

JoinDate should be stored as a datetime value.

Using `errors="coerce"` prevents invalid date strings from
causing the program to fail.

In [ ]:
customers["JoinDate"] = pd.to_datetime(
    customers["JoinDate"],
    errors="coerce"
)

customers["JoinDate"].dtype

dtype('<M8[us]')

## 16. Standardize City Values

City values should follow a consistent text format.

We will:

- Remove leading and trailing spaces
- Standardize capitalization
- Preserve `"Unknown"` values

This reduces inconsistent category names.

In [ ]:
customers["City"] = (
    customers["City"]
    .astype(str)
    .str.strip()
    .str.title()
)

print(
    customers["City"].value_counts().head(10)
)

City
Pokhara      66995
Nepalgunj    66897
Bharatpur    66821
Butwal       66734
Lalitpur     66709
Kathmandu    66697
Hetauda      66670
Birgunj      66588
Bhaktapur    66546
Ghorahi      66525
Name: count, dtype: int64


## 17. Standardize Gender Values

Gender is a categorical variable.

We will remove unnecessary spaces and standardize capitalization.

In [ ]:
customers["Gender"] = (
    customers["Gender"]
    .astype(str)
    .str.strip()
    .str.title()
)

print(
    customers["Gender"].value_counts(dropna=False)
)

Gender
Male      490527
Female    489408
Other      20065
Name: count, dtype: int64


## 18. Standardize Occupation Values

Occupation values will be standardized by removing unnecessary
spaces and applying consistent capitalization.

In [ ]:
customers["Occupation"] = (
    customers["Occupation"]
    .astype(str)
    .str.strip()
    .str.title()
)

print(
    customers["Occupation"].value_counts().head(15)
)

Occupation
Salaried         279573
Business         160092
Self-Employed    119127
Farmer           100001
Student           99995
Government        99340
Unemployed        79880
Retired           59992
Unknown            2000
Name: count, dtype: int64


## 19. Account Data Type Cleaning

We will ensure that account balances are numeric and OpenDate
is stored as a datetime value.

In [ ]:
accounts["Balance"] = pd.to_numeric(
    accounts["Balance"],
    errors="coerce"
)

accounts["OpenDate"] = pd.to_datetime(
    accounts["OpenDate"],
    errors="coerce"
)

print(accounts[["Balance", "OpenDate"]].dtypes)

Balance            float64
OpenDate    datetime64[us]
dtype: object


## 20. Standardize Account Type

AccountType is a categorical variable.

We will remove unnecessary spaces and standardize capitalization
so that equivalent categories are represented consistently.

In [ ]:
accounts["AccountType"] = (
    accounts["AccountType"]
    .astype(str)
    .str.strip()
    .str.title()
)

print(
    accounts["AccountType"].value_counts(dropna=False)
)

AccountType
Savings          715617
Current          233461
Salary           220913
Fixed Deposit    130009
Name: count, dtype: int64


## 21. Standardize Account Branch

Branch identifiers should use a consistent text representation.

In [ ]:
accounts["Branch"] = (
    accounts["Branch"]
    .astype(str)
    .str.strip()
)

print(
    accounts["Branch"].value_counts().head(10)
)

Branch
BR018    26384
BR011    26300
BR004    26262
BR046    26257
BR003    26199
BR047    26177
BR022    26177
BR037    26175
BR012    26148
BR030    26145
Name: count, dtype: int64


## 22. Transaction Data Type Cleaning

TransactionDate should be converted to datetime.

Amount should be converted to numeric.

Invalid values will be identified before they are handled.

In [ ]:
transactions["TransactionDate"] = pd.to_datetime(
    transactions["TransactionDate"],
    errors="coerce"
)

transactions["Amount"] = pd.to_numeric(
    transactions["Amount"],
    errors="coerce"
)

print(
    transactions[
        ["TransactionDate", "Amount"]
    ].dtypes
)

TransactionDate    datetime64[us]
Amount                    float64
dtype: object


## 23. Identify Invalid Transaction Amounts

Transaction amounts should represent valid financial transactions.

We will identify:

- Missing amounts
- Zero amounts
- Negative amounts

At this stage we are identifying invalid records before deciding
how they should be treated.

In [ ]:
invalid_transaction_amounts = transactions[
    transactions["Amount"].isna()
    | (transactions["Amount"] <= 0)
]

print(
    "Invalid transaction records:",
    invalid_transaction_amounts.shape[0]
)

display(
    invalid_transaction_amounts.head()
)

Invalid transaction records: 100


,TransactionID,AccountID,TransactionDate,TransactionType,Amount,Channel,Branch
89,90,2032440,2022-01-03,Card Payment,-1.0,Internet Banking,BR018
1792,1793,1438899,2022-08-18,Transfer Out,-1.0,ATM,BR031
1812,1813,1011781,2025-01-02,Withdrawal,-1.0,Mobile,BR016
6922,6923,1125399,2024-02-16,Deposit,-1.0,QR,BR045
7563,7564,2101438,2025-10-06,Transfer In,-1.0,ATM,BR012


## 24. Remove Invalid Transaction Amounts

Transactions with missing, zero, or negative amounts cannot be used
as valid financial transaction amounts.

These records will be removed from the cleaned transaction dataset.

The original transaction dataset remains unchanged.

In [ ]:
transactions = transactions[
    transactions["Amount"] > 0
].copy()

print(
    "Remaining invalid amounts:",
    (transactions["Amount"] <= 0).sum()
)

print(
    "Transactions after cleaning:",
    transactions.shape[0]
)

Remaining invalid amounts: 0
Transactions after cleaning: 4999900


## 25. Standardize Transaction Categories

TransactionType and Channel are categorical variables.

We will standardize their text representation.D

In [ ]:
transactions["TransactionType"] = (
    transactions["TransactionType"]
    .astype(str)
    .str.strip()
    .str.title()
)

transactions["Channel"] = (
    transactions["Channel"]
    .astype(str)
    .str.strip()
    .str.title()
)

print(
    transactions["TransactionType"].value_counts(
        dropna=False
    )
)

print(
    transactions["Channel"].value_counts(
        dropna=False
    )
)

TransactionType
Deposit         1100686
Withdrawal       998538
Transfer In      800811
Transfer Out     800456
Card Payment     799280
Bill Payment     500129
Name: count, dtype: int64
Channel
Mobile              1249084
Internet Banking     900633
Atm                  898711
Pos                  749858
Qr                   601405
Branch               600209
Name: count, dtype: int64


## 26. Loan Data Type Cleaning

LoanAmount, InterestRate, and RepaymentAmount are numerical fields.

LoanDate is a date field.

We will convert these columns to appropriate data types.

In [ ]:
loans["LoanAmount"] = pd.to_numeric(
    loans["LoanAmount"],
    errors="coerce"
)

loans["InterestRate"] = pd.to_numeric(
    loans["InterestRate"],
    errors="coerce"
)

loans["RepaymentAmount"] = pd.to_numeric(
    loans["RepaymentAmount"],
    errors="coerce"
)

loans["LoanDate"] = pd.to_datetime(
    loans["LoanDate"],
    errors="coerce"
)

print(
    loans[
        [
            "LoanAmount",
            "InterestRate",
            "RepaymentAmount",
            "LoanDate"
        ]
    ].dtypes
)

LoanAmount                float64
InterestRate              float64
RepaymentAmount           float64
LoanDate           datetime64[us]
dtype: object


## 27. Identify Unusual Loan Values

We will inspect loan amounts for values that may be invalid.

A loan amount less than or equal to zero is considered invalid
for this project.

We will also inspect interest rates and repayment amounts for
non-positive values.

These records will be reviewed before treatment.

In [ ]:
unusual_loans = loans[
    (loans["LoanAmount"] <= 0)
    | (loans["InterestRate"] <= 0)
    | (loans["RepaymentAmount"] < 0)
]

print(
    "Unusual loan records:",
    unusual_loans.shape[0]
)

display(
    unusual_loans.head()
)

Unusual loan records: 0


,LoanID,CustomerID,LoanType,LoanAmount,InterestRate,LoanDate,LoanStatus,RepaymentAmount


## 28. Remove Invalid Loan Records

Loan records with invalid financial values cannot be used
reliably for loan portfolio analysis.

We will remove records where:

- LoanAmount <= 0
- InterestRate <= 0
- RepaymentAmount < 0

In [ ]:
loans = loans[
    (loans["LoanAmount"] > 0)
    & (loans["InterestRate"] > 0)
    & (loans["RepaymentAmount"] >= 0)
].copy()

print(
    "Loans after cleaning:",
    loans.shape[0]
)

Loans after cleaning: 500000


## 29. Standardize Loan Categories

LoanType and LoanStatus are categorical variables.

We will remove unnecessary spaces and standardize capitalization.

In [ ]:
loans["LoanType"] = (
    loans["LoanType"]
    .astype(str)
    .str.strip()
    .str.title()
)

loans["LoanStatus"] = (
    loans["LoanStatus"]
    .astype(str)
    .str.strip()
    .str.title()
)

print(
    loans["LoanType"].value_counts(dropna=False)
)

print(
    loans["LoanStatus"].value_counts(dropna=False)
)

LoanType
Personal       140045
Home            90614
Business        84806
Auto            74399
Education       60085
Agriculture     50051
Name: count, dtype: int64
LoanStatus
Active       200430
Completed    149717
Approved      60036
Defaulted     49843
Rejected      39974
Name: count, dtype: int64


## 30. Branch Data Cleaning

The branches dataset contains information about bank branches.

We will standardize text fields and inspect missing values.

In [ ]:
print("Branch missing values before cleaning:")
print(branches.isna().sum())

Branch missing values before cleaning:
BranchID                  0
BranchName                0
City                      0
ManagerExperienceYears    0
OpenDate                  0
dtype: int64


In [ ]:
for column in branches.select_dtypes(
    include="object"
).columns:
    
    branches[column] = (
        branches[column]
        .astype(str)
        .str.strip()
        .str.title()
    )

print("Branch text fields standardized.")

Branch text fields standardized.


C:\Users\acer\AppData\Local\Temp\ipykernel_2444\3567324940.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in branches.select_dtypes(


## 31. Check Referential Integrity

The banking tables are related through key fields.

Expected relationships:

- Customers → Accounts through CustomerID
- Accounts → Transactions through AccountID
- Customers → Loans through CustomerID
- Accounts → Branches through Branch

We will identify records whose foreign keys do not exist
in the corresponding parent table.

In [ ]:
customer_ids = set(
    customers["CustomerID"].dropna()
)

account_ids = set(
    accounts["AccountID"].dropna()
)

branch_ids = set(
    branches["BranchID"].dropna()
)

account_customer_ids = set(
    accounts["CustomerID"].dropna()
)

transaction_account_ids = set(
    transactions["AccountID"].dropna()
)

loan_customer_ids = set(
    loans["CustomerID"].dropna()
)

account_branch_ids = set(
    accounts["Branch"].dropna()
)

print(
    "Accounts with unknown CustomerID:",
    len(account_customer_ids - customer_ids)
)

print(
    "Transactions with unknown AccountID:",
    len(transaction_account_ids - account_ids)
)

print(
    "Loans with unknown CustomerID:",
    len(loan_customer_ids - customer_ids)
)

print(
    "Accounts with unknown Branch:",
    len(account_branch_ids - branch_ids)
)

Accounts with unknown CustomerID: 0
Transactions with unknown AccountID: 0
Loans with unknown CustomerID: 0
Accounts with unknown Branch: 50


## 32. Final Missing Value Audit

After the cleaning operations, we will check the remaining
missing values in every dataset.

Remaining missing values are not automatically errors.
Some values may legitimately remain unavailable.

In [ ]:
for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    
    print(df.isna().sum())


Customers
CustomerID       0
Age              0
Gender           0
City             0
Occupation       0
Income        1000
JoinDate         0
dtype: int64

Accounts
AccountID      0
CustomerID     0
AccountType    0
Balance        0
OpenDate       0
Branch         0
dtype: int64

Transactions
TransactionID      0
AccountID          0
TransactionDate    0
TransactionType    0
Amount             0
Channel            0
Branch             0
dtype: int64

Loans
LoanID             0
CustomerID         0
LoanType           0
LoanAmount         0
InterestRate       0
LoanDate           0
LoanStatus         0
RepaymentAmount    0
dtype: int64

Branches
BranchID                  0
BranchName                0
City                      0
ManagerExperienceYears    0
OpenDate                  0
dtype: int64


In [ ]:
datasets = {
    "Customers": customers,
    "Accounts": accounts,
    "Transactions": transactions,
    "Loans": loans,
    "Branches": branches
}

print("Dataset dictionary updated with cleaned DataFrames.")

Dataset dictionary updated with cleaned DataFrames.


In [ ]:
for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print(df.isna().sum())


Customers
CustomerID       0
Age              0
Gender           0
City             0
Occupation       0
Income        1000
JoinDate         0
dtype: int64

Accounts
AccountID      0
CustomerID     0
AccountType    0
Balance        0
OpenDate       0
Branch         0
dtype: int64

Transactions
TransactionID      0
AccountID          0
TransactionDate    0
TransactionType    0
Amount             0
Channel            0
Branch             0
dtype: int64

Loans
LoanID             0
CustomerID         0
LoanType           0
LoanAmount         0
InterestRate       0
LoanDate           0
LoanStatus         0
RepaymentAmount    0
dtype: int64

Branches
BranchID                  0
BranchName                0
City                      0
ManagerExperienceYears    0
OpenDate                  0
dtype: int64


## 33. Final Duplicate Audit

We will verify whether duplicate records remain after cleaning.

This provides a final quality check before saving the cleaned data.


In [174]:
for name, df in datasets.items():
    print(
        name,
        "duplicate rows:",
        df.duplicated().sum()
    )

Customers duplicate rows: 0
Accounts duplicate rows: 0
Transactions duplicate rows: 0
Loans duplicate rows: 0
Branches duplicate rows: 0


## 34. Final Data Type Audit

We will verify the final data types of important columns
after the cleaning process.

In [175]:
print("CUSTOMERS")
print(customers.dtypes)

print("\nACCOUNTS")
print(accounts.dtypes)

print("\nTRANSACTIONS")
print(transactions.dtypes)

print("\nLOANS")
print(loans.dtypes)

print("\nBRANCHES")
print(branches.dtypes)

CUSTOMERS
CustomerID             int64
Age                    int64
Gender                   str
City                     str
Occupation               str
Income               float64
JoinDate      datetime64[us]
dtype: object

ACCOUNTS
AccountID               int64
CustomerID              int64
AccountType               str
Balance               float64
OpenDate       datetime64[us]
Branch                    str
dtype: object

TRANSACTIONS
TransactionID               int64
AccountID                   int64
TransactionDate    datetime64[us]
TransactionType               str
Amount                    float64
Channel                       str
Branch                        str
dtype: object

LOANS
LoanID                      int64
CustomerID                  int64
LoanType                      str
LoanAmount                float64
InterestRate              float64
LoanDate           datetime64[us]
LoanStatus                    str
RepaymentAmount           float64
dtype: object

BRANCHES


## 35. Compare Dataset Size Before and After Cleaning

We will compare the original row counts with the cleaned
row counts.

This helps us understand how many records were affected
by the cleaning process.

In [176]:
comparison = pd.DataFrame({
    "Dataset": [
        "Customers",
        "Accounts",
        "Transactions",
        "Loans",
        "Branches"
    ],
    
    "Original Rows": [
        customers_raw.shape[0],
        accounts_raw.shape[0],
        transactions_raw.shape[0],
        loans_raw.shape[0],
        branches_raw.shape[0]
    ],
    
    "Cleaned Rows": [
        customers.shape[0],
        accounts.shape[0],
        transactions.shape[0],
        loans.shape[0],
        branches.shape[0]
    ]
})

comparison["Rows Removed"] = (
    comparison["Original Rows"]
    - comparison["Cleaned Rows"]
)

comparison

,Dataset,Original Rows,Cleaned Rows,Rows Removed
0,Customers,1002000,1000000,2000
1,Accounts,1300000,1300000,0
2,Transactions,5000000,4999900,100
3,Loans,500000,500000,0
4,Branches,50,50,0


## 36. Save Cleaned Customers Dataset

The cleaned customer dataset will be saved separately from
the original raw dataset.

In [177]:
customers.to_csv(
    "../data/cleaned_dataset/customers_cleaned.csv",
    index=False
)

print("Cleaned customers dataset saved.")

Cleaned customers dataset saved.


## 37. Save Cleaned Accounts Dataset

In [178]:
accounts.to_csv(
    "../data/cleaned_dataset/accounts_cleaned.csv",
    index=False
)

print("Cleaned accounts dataset saved.")

Cleaned accounts dataset saved.


## 38. Save Cleaned Transactions Dataset

In [179]:
transactions.to_csv(
    "../data/cleaned_dataset/transactions_cleaned.csv",
    index=False
)

print("Cleaned transactions dataset saved.")

Cleaned transactions dataset saved.


## 39. Save Cleaned Loans Dataset

In [180]:
loans.to_csv(
    "../data/cleaned_dataset/loans_cleaned.csv",
    index=False
)

print("Cleaned loans dataset saved.")

Cleaned loans dataset saved.


## 40. Save Cleaned Branches Dataset

In [181]:
branches.to_csv(
    "../data/cleaned_dataset/branches_cleaned.csv",
    index=False
)

print("Cleaned branches dataset saved.")

Cleaned branches dataset saved.


## 41. Verify Cleaned Datasets

We will reload the saved cleaned datasets to confirm that
the files were successfully created and can be read correctly.

In [182]:
customers_check = pd.read_csv(
    "../data/cleaned_dataset/customers_cleaned.csv"
)

accounts_check = pd.read_csv(
    "../data/cleaned_dataset/accounts_cleaned.csv"
)

transactions_check = pd.read_csv(
    "../data/cleaned_dataset/transactions_cleaned.csv"
)

loans_check = pd.read_csv(
    "../data/cleaned_dataset/loans_cleaned.csv"
)

branches_check = pd.read_csv(
    "../data/cleaned_dataset/branches_cleaned.csv"
)

print("All cleaned datasets loaded successfully.")

All cleaned datasets loaded successfully.


## 42. Final Cleaning Validation

The cleaned datasets have been created and saved separately.

The final validation confirms:

- Raw data was loaded separately
- Working copies were used
- Duplicate customer IDs were handled
- Missing categorical values were handled
- Numerical data types were corrected
- Dates were standardized
- Categories were standardized
- Invalid transaction amounts were removed
- Unusual loan records were handled
- Table relationships were checked
- Cleaned datasets were saved separately

### Raw Data Principle

The files inside `data/raw_dataset/` remain unchanged.

The cleaned files are stored inside `data/cleaned_dataset/`.

The cleaned datasets are now ready for Exploratory Data Analysis.

In [183]:
final_datasets = {
    "Customers": customers_check,
    "Accounts": accounts_check,
    "Transactions": transactions_check,
    "Loans": loans_check,
    "Branches": branches_check
}

for name, df in final_datasets.items():
    print(
        f"{name}: "
        f"{df.shape[0]:,} rows × "
        f"{df.shape[1]} columns"
    )

print("\nPandas data cleaning completed successfully.")

Customers: 1,000,000 rows × 7 columns
Accounts: 1,300,000 rows × 6 columns
Transactions: 4,999,900 rows × 7 columns
Loans: 500,000 rows × 8 columns
Branches: 50 rows × 5 columns

Pandas data cleaning completed successfully.
